# NOVA AI — Local LLM on Kaggle (CPU)

A real local chat application powered by **Qwen2.5-3B-Instruct GGUF** and `llama-cpp-python`.

- No API key
- No GPU required (`n_gpu_layers=0`)
- Model weights are downloaded from Hugging Face
- Gradio chat UI with streaming responses and session history

**Before running:** In Kaggle Notebook settings, enable **Internet** and set **Accelerator = None**. The first run downloads a multi-GB model, so it may take longer than 10 minutes depending on Kaggle/network speed.

> No notebook can be guaranteed bug-free across all Kaggle images and package versions. This notebook includes checks and clear error messages; if a cell fails, share its full traceback.

## 1. Install dependencies

This installs the CPU runtime from the official `llama-cpp-python` CPU wheel index, plus Gradio and Hugging Face Hub.

In [ ]:
%pip -q install --upgrade gradio huggingface_hub

%pip -q install --upgrade llama-cpp-python   --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu

print("Dependencies installed. If Kaggle asks for a kernel restart, restart and continue from the next cell.")

## 2. Download the model

Repository: `Qwen/Qwen2.5-3B-Instruct-GGUF`

We select the Q4_K_M quantization to reduce RAM use. The exact filename is checked against the repository; if the repository changes, the error lists available GGUF files.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import hf_hub_download, HfApi

MODEL_REPO = "Qwen/Qwen2.5-3B-Instruct-GGUF"
MODEL_FILENAME = "qwen2.5-3b-instruct-q4_k_m.gguf"
MODEL_DIR = Path("/kaggle/working/nova_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / MODEL_FILENAME

if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 100_000_000:
    print(f"Downloading {MODEL_FILENAME} ...")
    try:
        downloaded = hf_hub_download(
            repo_id=MODEL_REPO,
            filename=MODEL_FILENAME,
            local_dir=str(MODEL_DIR),
        )
        MODEL_PATH = Path(downloaded)
    except Exception as exc:
        print("Could not download the configured filename.")
        try:
            files = HfApi().list_repo_files(MODEL_REPO)
            gguf_files = [f for f in files if f.lower().endswith(".gguf")]
            print("GGUF files currently present in the repository:")
            for name in gguf_files[:40]:
                print(" -", name)
        except Exception:
            pass
        raise RuntimeError(
            "Model download failed. Check Kaggle Internet access and the repository filename above."
        ) from exc

if not MODEL_PATH.exists() or MODEL_PATH.stat().st_size < 100_000_000:
    raise RuntimeError(f"Model file is missing or unexpectedly small: {MODEL_PATH}")

print(f"Model ready: {MODEL_PATH}")
print(f"Size: {MODEL_PATH.stat().st_size / (1024**3):.2f} GiB")

## 3. Load the actual model on CPU

This is the model inference engine. `n_gpu_layers=0` explicitly disables GPU offload.

In [ ]:
import os
from llama_cpp import Llama

CPU_THREADS = max(1, (os.cpu_count() or 2) - 1)

print(f"Loading model with {CPU_THREADS} CPU threads...")
llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=2048,
    n_threads=CPU_THREADS,
    n_batch=128,
    n_gpu_layers=0,
    use_mmap=True,
    verbose=False,
)

print("NOVA model loaded successfully.")

## 4. Build the chat application

Conversation history is held in the Gradio session. It lasts while the running notebook/app remains alive; it is not persistent storage across notebook restarts.

In [ ]:
import gradio as gr

SYSTEM_PROMPT = (
    "You are NOVA AI, a helpful, accurate and friendly assistant. "
    "You can communicate in Persian and English. "
    "Reply in the same language as the user. "
    "Use clear explanations and Markdown when useful. "
    "Do not claim to have performed actions you did not perform."
)

def chat_fn(message, history, max_tokens, temperature):
    message = (message or "").strip()
    if not message:
        yield ""
        return

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Gradio ChatInterface supplies history as role/content message dictionaries.
    for item in (history or [])[-12:]:
        if not isinstance(item, dict):
            continue
        role = item.get("role")
        content = item.get("content")
        if role in ("user", "assistant") and isinstance(content, str) and content.strip():
            messages.append({"role": role, "content": content})

    messages.append({"role": "user", "content": message})

    answer = ""
    try:
        stream = llm.create_chat_completion(
            messages=messages,
            max_tokens=int(max_tokens),
            temperature=float(temperature),
            top_p=0.9,
            repeat_penalty=1.1,
            stream=True,
        )
        for chunk in stream:
            choices = chunk.get("choices") or []
            if not choices:
                continue
            delta = choices[0].get("delta") or {}
            token = delta.get("content")
            if isinstance(token, str) and token:
                answer += token
                yield answer

        if not answer:
            yield "(The model returned an empty response. Try another prompt.)"

    except Exception as exc:
        yield (
            "NOVA encountered an inference error.\\n\\n"
            f"`{type(exc).__name__}: {str(exc)[:700]}`"
        )

with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="violet", neutral_hue="slate"),
    title="NOVA AI — Local Qwen",
    css="""
    .gradio-container { max-width: 1100px !important; margin: auto !important; }
    footer { display: none !important; }
    """,
) as demo:
    gr.Markdown(
        "# ✦ NOVA AI\\n"
        "### Local language model · Qwen2.5-3B-Instruct · CPU inference"
    )

    gr.ChatInterface(
        fn=chat_fn,
        chatbot=gr.Chatbot(
            type="messages",
            height=520,
            show_copy_button=True,
            placeholder="Start a conversation with NOVA AI.",
        ),
        textbox=gr.Textbox(
            placeholder="Message NOVA AI… / پیام خود را بنویسید…",
            lines=2,
            max_lines=5,
        ),
        additional_inputs=[
            gr.Slider(
                minimum=64, maximum=512, value=256, step=32,
                label="Maximum new tokens"
            ),
            gr.Slider(
                minimum=0.1, maximum=1.2, value=0.7, step=0.1,
                label="Temperature"
            ),
        ],
        examples=[
            "سلام! خودت را معرفی کن.",
            "Explain how a neural network learns.",
            "یک تابع پایتون برای تشخیص عدد اول بنویس.",
        ],
        cache_examples=False,
    )

print("NOVA AI interface is ready to launch.")

## 5. Launch the app

Kaggle can display the app inline. `share=True` also requests a temporary public Gradio URL. Keep this cell running while using the app.

In [ ]:
demo.queue().launch(
    share=True,
    inline=True,
    show_error=True,
)

## Troubleshooting

- **Internet/download error:** confirm Internet is enabled in Kaggle settings.
- **Package install error:** rerun the install cell; if it persists, capture the complete pip error and the Kaggle Python version.
- **Out of memory:** use a smaller quantization/model (for example a 1.5B model) or reduce `n_ctx` to 1024 and `n_batch` to 64.
- **Slow answers:** expected on CPU. Lower “Maximum new tokens” to 128.
- **Model filename not found:** the download cell prints available `.gguf` files so the filename can be updated.
- **Temporary link stops working:** the Kaggle session or Gradio share session has stopped; rerun the launch cell.

This is a runnable starter notebook, not a guarantee of zero failures: Kaggle images, package wheels, model hosting and runtime quotas can change.